# Day 4: 扩散模型核心原理

## 🎯 本节目标

完成今天的学习后，你将能够：

1. ✅ 理解扩散模型的直观原理
2. ✅ 掌握 Add-THIN 时间扩散模型
3. ✅ 理解离散空间扩散模型
4. ✅ 掌握条件生成机制

---

## 📚 什么是扩散模型？

### 直观理解

**扩散模型**是一种生成模型，核心思想是：

```
训练（加噪）          推理（去噪）

清晰图像      →      加噪      →      纯噪声
   ↑                                  ↓
   └──────── 反向去噪 ←───────────────┘
            （学习这个过程）
```

### 前端类比

| 扩散模型概念 | 前端类比 |
|-------------|----------|
| 加噪过程 | 图片逐渐模糊（CSS filter） |
| 去噪过程 | 逐步恢复清晰度 |
| 生成新样本 | 从随机 seed 生成新图片 |

---

## Part 1: 扩散模型基础

### 1.1 DDPM 框架

**DDPM** (Denoising Diffusion Probabilistic Models) 是最经典的扩散模型。

#### 前向过程（加噪）

```python
# 逐步加噪
x_0 → x_1 → x_2 → ... → x_T (纯噪声)

# 每一步加一点高斯噪声
x_t = sqrt(alpha_t) * x_0 + sqrt(1 - alpha_t) * epsilon
```

#### 反向过程（去噪）

```python
# 逐步去噪
x_T (纯噪声) → x_{T-1} → ... → x_1 → x_0 (清晰数据)

# 每一步预测并去除噪声
x_{t-1} = predict(x_t, t)  # 使用神经网络预测
```

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np

print("🎲 扩散过程可视化")
print("="*50)

# 设置随机种子
torch.manual_seed(42)

# 模拟数据：一条简单的正弦曲线
x0 = torch.sin(torch.linspace(0, 4*np.pi, 100))

# 定义噪声调度（逐步增加噪声）
T = 10  # 扩散步数
betas = torch.linspace(0.0001, 0.02, T)
alphas = 1 - betas
alpha_bars = torch.cumprod(alphas, dim=0)

# 存储每一步的结果
noisy_versions = [x0]
x_current = x0.clone()

# 前向扩散（加噪）
for t in range(T):
    noise = torch.randn_like(x_current)
    alpha_bar = alpha_bars[t]
    
    # 加噪公式
    x_current = torch.sqrt(alpha_bar) * x0 + torch.sqrt(1 - alpha_bar) * noise
    noisy_versions.append(x_current.clone())

# 可视化
fig, axes = plt.subplots(2, 6, figsize=(18, 6))
axes = axes.flatten()

# 只显示前 11 个时间步（包括原始数据）
for i in range(min(11, len(noisy_versions))):
    ax = axes[i]
    ax.plot(noisy_versions[i].numpy(), linewidth=2)
    ax.set_title(f't={i}' if i == 0 else f't={i} (ᾱ={alpha_bars[i-1]:.3f})')
    ax.set_ylim(-3, 3)
    ax.grid(True, alpha=0.3)

# 隐藏多余的子图
for i in range(len(noisy_versions), len(axes)):
    axes[i].set_visible(False)

plt.tight_layout()
plt.show()

print("\n观察要点:")
print("1. t=0: 原始清晰数据（正弦波）")
print("2. t=1~9: 逐步加噪，信号逐渐模糊")
print("3. t=10: 几乎变成纯噪声（正弦波完全消失）")

---

## Part 2: Add-THIN 时间扩散模型

### 2.1 为什么需要 Add-THIN？

**传统 DDPM** 处理的是**连续值**（如图像像素），但**时间序列数据**有不同的特性：

| 特性 | 图像数据 | 时间序列数据 |
|------|----------|--------------|
| 数据类型 | 连续值（像素） | 离散事件点 |
| 数据结构 | 网格 | 事件序列 |
| 噪声类型 | 高斯噪声 | 事件增删 |

**Add-THIN** 是专门为**时间点过程**设计的扩散模型。

### 2.2 Add-THIN 核心思想

```
前向扩散（加噪）:
1. THIN:  随机删除一些事件点
2. ADD:   添加一些噪声事件点（来自泊松过程）

反向扩散（去噪）:
1. 预测哪些事件是真实的
2. 逐步恢复原始事件序列
```

In [ ]:
import torch

print("⏱️ Add-THIN 模拟")
print("="*50)

# 原始时间序列（事件发生的时间点）
x0 = torch.tensor([1.0, 3.0, 5.0, 7.0, 9.0])
tmax = 10.0

print(f"原始序列 x0: {x0}")
print(f"时间范围: [0, {tmax}]")

# 模拟 THIN 操作（删除事件）
def thin_process(x, alpha=0.5):
    """
    THIN 操作：以概率 alpha 保留每个事件
    
    alpha: 保留概率（越小，删除越多）
    """
    keep_mask = torch.rand(len(x)) < alpha
    return x[keep_mask], keep_mask

# 模拟 ADD 操作（添加噪声事件）
def add_process(x, num_add=2, tmax=10.0):
    """
    ADD 操作：添加来自泊松过程的事件
    
    num_add: 要添加的事件数量
    tmax: 最大时间
    """
    new_events = torch.rand(num_add) * tmax
    return torch.cat([x, new_events])

# 演示扩散过程
print("\n扩散过程演示:")
x_current = x0.clone()

for step in range(4):
    print(f"\n步骤 {step}:")
    print(f"  当前序列: {torch.sort(x_current)[0]}")
    
    # THIN 操作
    alpha = 0.7 ** step  # 保留概率逐渐降低
    x_kept, keep_mask = thin_process(x_current, alpha=alpha)
    print(f"  THIN (α={alpha:.2f}): 保留 {x_kept}")
    
    # ADD 操作
    num_add = step + 1
    x_next = add_process(x_kept, num_add=num_add, tmax=tmax)
    x_next = torch.sort(x_next)[0]
    print(f"  ADD (+{num_add}): 添加后 {x_next}")
    
    x_current = x_next

print("\n" + "="*50)
print("最终结果: 纯噪声（均匀分布的事件）")

### 2.3 Add-THIN 数学原理

#### 前向扩散公式

```python
# 给定原始序列 x_0，计算第 n 步的 x_n

# 1. THIN 操作
alpha_bar_n = cumulative_product(alpha)  # 累积保留概率
x_0_kept, x_0_thinned = x_0.thin(alpha=alpha_bar_n[n])

# 2. ADD 操作
# 生成 HPP（齐次泊松过程）事件
hpp = HomogeneousPoissonProcess(rate=lambda, tmax=tmax)
x_n = x_0_kept.add_events(hpp)
```

#### 反向扩散公式

```python
# 给定 x_n，预测 x_{n-1}

# 1. 使用神经网络预测
log_p_keep = model(x_n, n)  # 预测每个事件的保留概率

# 2. 采样决定保留还是删除
keep_mask = sample_bernoulli(log_p_keep)
x_{n-1} = x_n[keep_mask]

# 3. 添加一些事件（逆转 THIN）
new_events = sample_from_model(x_{n-1}, n)
x_{n-1} = x_{n-1}.add_events(new_events)
```

---

## Part 3: 离散空间扩散模型

### 3.1 为什么需要离散扩散？

**时间维度**：使用 Add-THIN（连续时间）
**空间维度**：使用离散扩散（离散 POI）

POI 是**离散类别**（如 POI #123），不能直接加高斯噪声！

### 3.2 离散扩散原理

#### 前向扩散：逐步混合类别分布

```python
# 原始分布：某个 POI 的概率为 1
P(x_0 = POI_123) = 1.0
P(x_0 = other) = 0.0

# 逐步扩散：混合到均匀分布
P(x_t | x_0) = (1 - alpha_t) * P(x_0) + alpha_t * P(uniform)

# 最终分布：所有 POI 概率相等
P(x_T) = uniform
```

In [ ]:
import torch
import torch.nn.functional as F

print("🎯 离散扩散模拟")
print("="*50)

# 假设有 5 个 POI 类别
num_pois = 5

# 原始分布：POI #2 的概率为 1
x0 = F.one_hot(torch.tensor([2]), num_classes=num_pois).float()
print(f"原始分布 x0: {x0}")
print(f"(POI #{x0.argmax().item()} 的概率为 1)\n")

# 扩散调度
T = 10
alphas = torch.linspace(0, 1, T)  # 从 0 到 1

# 前向扩散
distributions = [x0.squeeze()]
current_dist = x0.squeeze()

print("前向扩散过程:")
for t in range(1, T):
    alpha = alphas[t]
    uniform = torch.ones(num_pois) / num_pois
    
    # 混合公式
    current_dist = (1 - alpha) * distributions[0] + alpha * uniform
    distributions.append(current_dist.clone())
    
    if t % 2 == 0 or t == T - 1:
        print(f"  t={t}: α={alpha:.2f}, max_prob={current_dist.max():.3f}")

# 可视化
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.flatten()

for i in range(T):
    ax = axes[i]
    ax.bar(range(num_pois), distributions[i].numpy())
    ax.set_title(f't={i}')
    ax.set_ylim(0, 1)
    ax.set_xlabel('POI ID')
    if i % 5 == 0:
        ax.set_ylabel('Probability')

plt.tight_layout()
plt.show()

print("\n观察要点:")
print("1. t=0: 所有概率集中在 POI #2")
print("2. t=1~9: 概率逐渐扩散到其他 POI")
print("3. t=9: 所有 POI 概率几乎相等（均匀分布）")

### 3.3 Gumbel Softmax 采样

**问题**：离散分布无法直接梯度反向传播

**解决**：Gumbel Softmax（连续松弛）

```python
# 标准 softmax（不可微）
sample = argmax(softmax(logits))

# Gumbel Softmax（可微）
# 添加 Gumbel 噪声后用 softmax，温度 τ 控制离散程度
sample = gumbel_softmax_sample(logits, temperature=tau)
# 当 tau → 0，逼近真正的离散采样
```

In [ ]:
def gumbel_softmax_sample(logits, temperature=1.0):
    """
    Gumbel Softmax 采样
    
    Args:
        logits: [num_classes] 未归一化的对数概率
        temperature: 温度参数（越小越离散）
    
    Returns:
        [num_classes] 采样后的概率分布
    """
    # 1. 生成 Gumbel 噪声
    gumbel_noise = -torch.log(-torch.log(torch.rand_like(logits) + 1e-10) + 1e-10)
    
    # 2. 添加噪声并除以温度
    y = (logits + gumbel_noise) / temperature
    
    # 3. Softmax
    return F.softmax(y, dim=-1)

# 演示不同温度下的采样
logits = torch.tensor([2.0, 1.0, 0.5, 0.2, 0.1])  # POI #0 概率最高

print("🌡️ Gumbel Softmax 温度演示")
print("="*50)
print(f"原始 logits: {logits}\n")

temperatures = [1.0, 0.5, 0.1]

for temp in temperatures:
    sample = gumbel_softmax_sample(logits, temperature=temp)
    print(f"温度 τ={temp}: {sample.numpy()}")
    print(f"  采样结果: POI #{sample.argmax().item()} (概率={sample.max().item():.3f})\n")

print("观察要点:")
print("- τ=1.0: 概率分布较平滑（不确定性高）")
print("- τ=0.5: 概率分布更尖锐（逐渐收敛）")
print("- τ=0.1: 几乎是 one-hot（近似离散采样）")

---

## Part 4: 条件生成机制

### 4.1 什么是条件生成？

**条件生成**：根据给定条件生成数据。

在 Marionette 中：

```
条件 (Condition) → 生成轨迹 (Trajectory)

例如:
  - 星期几 = 周一
  - 是否斋月 = 否
  - 是否假期 = 否
  → 生成符合这些条件的轨迹
```

### 4.2 条件编码

In [ ]:
import torch
import torch.nn as nn

print("🎛️ 条件编码演示")
print("="*50)

# 定义条件
conditions = {
    'day_of_week': 25,  # 周一
    'ramadan': 32,      # 非斋月
    'holiday': 34       # 非假期
}

print("输入条件:")
for k, v in conditions.items():
    print(f"  {k}: {v}")

# 方法 1: One-hot 编码
day_embedding_onehot = F.one_hot(torch.tensor(conditions['day_of_week'] - 25), num_classes=7).float()
print(f"\nOne-hot 编码 (星期): {day_embedding_onehot}")

# 方法 2: Embedding 层（可学习）
class ConditionEncoder(nn.Module):
    def __init__(self, num_conditions=6, embed_dim=32):
        super().__init__()
        # 每个条件类型有自己的嵌入层
        self.day_embed = nn.Embedding(7, embed_dim)  # 7 天
        self.bool_embed = nn.Embedding(2, embed_dim)  # 2 值
        
        # 最终投影
        self.fc = nn.Linear(embed_dim * 3, embed_dim)
    
    def forward(self, day, ramadan, holiday):
        # 编码每个条件
        day_emb = self.day_embed(day - 25)  # 转换为 0-6
        ramadan_emb = self.bool_embed(ramadan - 32)  # 转换为 0-1
        holiday_emb = self.bool_embed(holiday - 34)  # 转换为 0-1
        
        # 拼接
        combined = torch.cat([day_emb, ramadan_emb, holiday_emb], dim=-1)
        
        # 投影
        output = self.fc(combined)
        return output

# 使用编码器
encoder = ConditionEncoder()
condition_emb = encoder(
    torch.tensor([conditions['day_of_week']]),
    torch.tensor([conditions['ramadan']]),
    torch.tensor([conditions['holiday']])
)

print(f"\n条件嵌入维度: {condition_emb.shape}")
print(f"条件嵌入: {condition_emb}")

### 4.3 条件如何影响生成？

```python
# 训练时：条件作为额外输入
def forward(self, x_t, t, conditions):
    # 1. 编码条件
    cond_emb = self.condition_encoder(conditions)
    
    # 2. 编码时间步
    t_emb = self.time_encoder(t)
    
    # 3. 拼接嵌入
    emb = cond_emb + t_emb
    
    # 4. 预测噪声
    noise_pred = self.network(emb, x_t)
    return noise_pred

# 生成时：提供目标条件
def sample(self, conditions):
    # 从纯噪声开始
    x_T = torch.randn(...)
    
    # 逐步去噪（每步都使用条件）
    for t in reversed(range(T)):
        x_{t-1} = self.denoise(x_t, t, conditions)
    
    return x_0
```

---

## Part 5: Marionette 的混合架构

### 5.1 整体架构图

```
输入: 条件 (星期、斋月、假期...)
    ↓
┌─────────────────────────────────────┐
│                                   │
│  时间扩散 (Add-THIN)  空间扩散 (离散)│
│       ↓                  ↓         │
│  生成时间序列          生成 POI 序列  │
│       ↓                  ↓         │
│    tau[], time[]     category[], POI[]│
│                                   │
└─────────────────────────────────────┘
    ↓
合并: 完整轨迹 (时间 + POI)
```

### 5.2 联合生成流程

In [ ]:
print("🔗 Marionette 联合生成流程")
print("="*50)

print("\n步骤 1: 生成时间序列")
print("- 输入: 条件 (星期、斋月、假期)")
print("- 模型: Add-THIN (时间扩散模型)")
print("- 输出: 时间间隔 tau[] 和 绝对时间 time[]")

print("\n步骤 2: 生成 POI 序列")
print("- 输入: 时间序列 + 条件")
print("- 模型: DiffusionTransformer (空间扩散模型)")
print("- 输出: POI 类别 category[] 和 POI ID checkins[]")

print("\n步骤 3: 组合完整轨迹")
print("```")
trajectory = [
    {"time": 3600, "poi": 10, "category": 0},
    {"time": 7200, "poi": 45, "category": 2},
    {"time": 10800, "poi": 23, "category": 1},
]
print("```")

print("\n优势:")
print("1. 时间和空间独立建模，各自最优")
print("2. 可以分别调整两个模型的参数")
print("3. 生成质量更高")

---

## Part 6: 实践练习

### ✅ 练习 1: 实现简单的扩散过程

In [ ]:
# TODO: 实现简单的一维扩散过程

def forward_diffusion(x0, num_steps, alpha_schedule):
    """
    前向扩散过程
    
    Args:
        x0: 原始数据 [batch_size]
        num_steps: 扩散步数
        alpha_schedule: 调度参数 [num_steps]
    
    Returns:
        List[Tensor]: 每一步的中间结果
    """
    # 实现你的代码
    pass

def reverse_diffusion(xT, num_steps, model, alpha_schedule):
    """
    反向扩散过程
    
    Args:
        xT: 纯噪声 [batch_size]
        num_steps: 扩散步数
        model: 去噪模型
        alpha_schedule: 调度参数 [num_steps]
    
    Returns:
        Tensor: 生成的数据
    """
    # 实现你的代码
    pass

print("请实现扩散过程函数...")

### ✅ 练习 2: 模拟 Add-THIN 的 thin 和 add 操作

In [ ]:
# TODO: 实现 Add-THIN 的核心操作

def thin_events(events, keep_prob):
    """
    THIN 操作：随机删除事件
    
    Args:
        events: 事件时间 [num_events]
        keep_prob: 保留概率
    
    Returns:
        kept_events: 保留的事件
        removed_events: 删除的事件
    """
    # 实现你的代码
    pass

def add_events(existing_events, num_add, tmax):
    """
    ADD 操作：添加泊松过程事件
    
    Args:
        existing_events: 现有事件
        num_add: 要添加的事件数
        tmax: 最大时间
    
    Returns:
        combined_events: 合并后的事件（已排序）
    """
    # 实现你的代码
    pass

print("请实现 Add-THIN 操作...")

---

## 📝 Day 4 总结

### 今天学到了什么？

#### ✅ 扩散模型基础
- 前向扩散（加噪）和反向扩散（去噪）
- DDPM 框架的基本原理

#### ✅ Add-THIN 时间扩散
- THIN 操作（删除事件）
- ADD 操作（添加事件）
- 适用于时间点过程

#### ✅ 离散空间扩散
- 类别分布的逐步混合
- Gumbel Softmax 采样
- 适用于离散 POI 序列

#### ✅ 条件生成
- 条件编码方法
- 条件如何影响生成

---

### 🎯 明天预告：训练与采样复现

Day 5 将学习：

1. 训练流程的完整实现
2. 采样流程的详细步骤
3. W&B 实验跟踪
4. 实际运行训练

---

**🎉 恭喜完成 Day 4！**

明天我们将进入实战环节，真正运行训练和采样。